In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, Layout, VBox
from IPython.display import display

def plot_filter_response(filter_type='Low-pass', Q=5.0):
    fig, ax = plt.subplots(figsize=(9, 6))
    
    # Συχνότητες κανονικοποιημένες γύρω από το f = 1 Hz (omega / omega_0)
    freq = np.logspace(-1, 1, 1000) # από 0.1 έως 10 Hz
    omega = freq # Κανονικοποιημένη συχνότητα ω/ω_0 = f/f_c
    
    if filter_type == 'Low-pass':
        # Gain magnitude για Low-pass 2ης τάξης: H = 1 / sqrt((1 - omega^2)^2 + (omega/Q)^2)
        H_mag = 1.0 / np.sqrt((1.0 - omega**2)**2 + (omega / Q)**2)
        title_str = "Low-pass filter"
    elif filter_type == 'High-pass':
        # Gain magnitude για High-pass 2ης τάξης: H = omega^2 / sqrt((1 - omega^2)^2 + (omega/Q)^2)
        H_mag = omega**2 / np.sqrt((1.0 - omega**2)**2 + (omega / Q)**2)
        title_str = "High-pass filter"
    elif filter_type == 'Band-pass':
        # Gain magnitude για Band-pass 2ης τάξης: H = (omega/Q) / sqrt((1 - omega^2)^2 + (omega/Q)^2)
        H_mag = (omega / Q) / np.sqrt((1.0 - omega**2)**2 + (omega / Q)**2)
        title_str = "Band-pass filter"
    elif filter_type == 'Band-stop':
        # Gain magnitude για Band-stop (Notch) 2ης τάξης: H = |1 - omega^2| / sqrt((1 - omega^2)^2 + (omega/Q)^2)
        H_mag = np.abs(1.0 - omega**2) / np.sqrt((1.0 - omega**2)**2 + (omega / Q)**2)
        title_str = "Band-stop filter"

    # Μετατροπή σε dB
    gain_db = 20 * np.log10(np.clip(H_mag, 1e-5, 100))

    ax.semilogx(freq, gain_db, 'r-', linewidth=1.5)

    # Γραφική διακόσμηση αντίστοιχη των διαγραμμάτων
    ax.set_xlim(0.1, 10)
    ax.set_ylim(-70, 30 if filter_type in ['Low-pass', 'High-pass'] else 10)
    ax.set_xlabel('Frequency (Hz)', fontsize=12)
    ax.set_ylabel('Gain (dB)', fontsize=12)
    ax.grid(True, which="both", ls=":", color='gray', alpha=0.7)
    
    # Κύριες γραμμές αναφοράς
    ax.axvline(1.0, color='gray', linestyle='-', linewidth=0.8)
    ax.axhline(0.0, color='gray', linestyle='-', linewidth=0.8)

    ax.set_xticks([0.1, 1, 10])
    ax.set_xticklabels(['0.1', '1', '10'])
    
    # Ετικέτα τύπου φίλτρου στο κάτω μέρος
    ax.set_title(f"Gain response of {title_str.lower()} with Q = {Q}", fontsize=13, pad=12)
    
    plt.tight_layout()
    plt.show()

# Ορισμός widgets
q_slider = FloatSlider(
    min=0.1, max=20.0, step=0.1, value=5.0, 
    description='Q Factor:', 
    style={'description_width': '80px'}, 
    layout=Layout(width='280px')
)

filter_radio = RadioButtons(
    options=['Low-pass', 'High-pass', 'Band-pass', 'Band-stop'],
    value='Low-pass',
    description='<b>Filter Type:</b>',
    disabled=False,
    layout=Layout(width='220px', height='140px')
)

# Δημιουργία διαδραστικού περιβάλλοντος
interactive_plot = interactive(plot_filter_response, filter_type=filter_radio, Q=q_slider)

# Διάταξη: Σχήμα αριστερά, Controls (Slider & Radio buttons κάθετα) στα δεξιά
controls_box = VBox([interactive_plot.children[0], interactive_plot.children[1]], 
                    layout=Layout(justify_content='center', align_items='flex-start', margin='0 0 0 20px'))

plot_output = interactive_plot.children[-1]

main_layout = HBox([plot_output, controls_box], layout=Layout(align_items='center'))

display(main_layout)